# Reproducing Martel et al. (2025) from dandiset 001174: sequences of calcium events (Figure 4)

Paper. Martel A-C et al. (2025), *iScience* 28, 112767,
[doi:10.1016/j.isci.2025.112767](https://doi.org/10.1016/j.isci.2025.112767); claims file
`claims/001174/10.1016-j.isci.2025.112767.yaml`, written from the version of record (Europe PMC,
PMC12192352) without the supplement.

Dandiset. [DANDI:001174](https://dandiarchive.org/dandiset/001174/0.250331.2218), version
0.250331.2218, the version the paper cites and the only published one (see the first notebook,
`reproduce_martel_2025.ipynb`, for the data check and the choice of version).

Method. Method S of the claims file, a port of the paper's `identify_sequences.m` (Zenodo
doi:10.5281/zenodo.14963234) in `sequences.py`, run on the dandiset's deconvolved events. Events are
2 to 5 events of different cells, each within 2 s of the one before (Methods, "Determination of
precisely timed sequences"); kept when repeated at least 4 times in 10 min (Methods), above the 0.95
centile of 1000 circularly shifted shuffles (Methods; the code's centile). The claims file names the
readings the paper leaves open, all run here:

- the two 0.05 s rules, reading (a) both, (b) a 0.05 s minimum between successive events of a
  sequence only, (c) a 0.05 s minimum between the starts of repetitions only;
- which 10 min of the task: (i) the first 10 min, (ii) the last 10 min (spontaneous: the first 10
  min, and the next 10 min where it lasts 20 min);
- variants: the code's rule of more than 4 repetitions, single-cell repeats included, every ordering
  of the cells (reading a only, for the example sessions), and the double-counting step as its
  comments describe it.

The main count uses the double-counting step as the code runs it: the step marks used events by
writing 2 into a logical array, which MATLAB stores as true, so its test for 2 never succeeds and
no repetition is removed (see `sequences.py`). The claims file says "double counting removed as the
code does"; the documented step is shown as a variant.

| Claim | Tested here |
|---|---|
| fig4a-example-sequences-q | tested |
| fig4b-example-three-segments-q | tested |
| fig4d-sequence-measures-vary | tested |
| fig4c-sequence-cells-not-clustered | tested |

In [ ]:
import time
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import sma_m1 as h
import sequences as sq

pd.set_option("display.width", 200)
print("dandiset", h.DANDISET, "version", h.VERSION)
files = h.load_all()
files = [x for x in files if not (x["subject"] == "F" and x["date"] == "2024-03-19")]
print(len(files), "files (F 2024-03-19, one cell, is left out, as the Jaccard code leaves it out)")

## Data check

Per file, the events the sequence search uses: cells with any event, events, and the 10-min slices.

In [ ]:
rows = []
for x in files:
    times, keep = sq.event_times(x)
    for tr in (["i", "ii"] if x["condition"] == "task" else ["i"]):
        for name, ts in sq.segments(x, tr):
            n_ev = sum(((t > ts[0]) & (t <= ts[1])).sum() for t in times)
            rows.append({"subject": x["subject"], "region": x["region"], "date": x["date"], "condition": x["condition"],
                         "task_reading": tr, "segment": name, "start_s": round(ts[0], 1), "end_s": round(ts[1], 1),
                         "cells": x["events"].shape[1], "cells_with_events": len(keep), "events": int(n_ev)})
seg = pd.DataFrame(rows)
print(seg.to_string())
print("segments:", len(seg), "| shorter than 10 min:", int(((seg.end_s - seg.start_s) < 599).sum()))

## Detection on every slice

One run per slice and reading of the 0.05 s rules, each with its own seeded generator. The task's reading (ii) reuses reading (i) where the file is 10 min or shorter.

In [ ]:
t0 = time.time()
RES = {}
for x in files:
    for tr in (["i", "ii"] if x["condition"] == "task" else ["i"]):
        for name, ts in sq.segments(x, tr):
            for rd in sq.READINGS:
                key = (x["subject"], x["region"], x["date"], x["condition"], tr, name, rd)
                prev = (x["subject"], x["region"], x["date"], x["condition"], "i", name, rd)
                if tr == "ii" and sq.segments(x, "i")[0][1] == ts:
                    RES[key] = RES[prev]
                    continue
                seed = int(pd.util.hash_pandas_object(pd.Series([str(key)])).iloc[0] % 2**32)
                RES[key] = sq.run_segment(x, ts, rd, np.random.default_rng(seed))
print("runs:", len(RES), "in %.0f s" % (time.time() - t0))

In [ ]:
def measures(key, **kw):
    sig, info = RES[key]
    x = [f for f in files if (f["subject"], f["region"], f["date"], f["condition"]) == key[:4]][0]
    seqs = sq.finalize(sig, **kw)
    s = sq.summary(seqs, x["events"].shape[1])
    s.update({"candidates": info["candidates"], "passing_reps": info["passing_reps"], "significant": info["significant"]})
    return s, seqs

VARIANTS = {"main": {}, "rule >": {"rule": ">"}, "single-cell": {"single_cell": True}, "documented dedupe": {"dedupe": "documented"}}
allrows = []
for key in RES:
    for vname, kw in VARIANTS.items():
        s, _ = measures(key, **kw)
        allrows.append(dict(zip(["subject", "region", "date", "condition", "task_reading", "segment", "reading"], key), variant=vname, **s))
ALL = pd.DataFrame(allrows)
print(ALL[ALL.variant == "main"].groupby(["reading", "task_reading", "segment"]).agg(
    slices=("n_seq", "size"), median_n_seq=("n_seq", "median"), max_n_seq=("n_seq", "max"),
    median_significant=("significant", "median")).to_string())

## fig4a-example-sequences-q

> In an example session of monkey Q's SMA with 40 cells, 13 sequences were detected in the
> spontaneous condition and 17 during the task, and the specific sequences differed between the two
> even where they involved the same cells.

The sessions of Q's SMA with 40 accepted cells are the candidates (Figure 4A legend).

In [ ]:
cand = sorted({(x["date"]) for x in files if x["subject"] == "Q" and x["events"].shape[1] == 40})
print("Q SMA sessions with 40 accepted cells:", cand)
out = []
for d in cand:
    for rd in sq.READINGS:
        for tr in ("i", "ii"):
            for vname, kw in VARIANTS.items():
                ks = ("Q", "SMA", d, "spontaneous", "i", "S1", rd)
                kt = ("Q", "SMA", d, "task", tr, "RT", rd)
                (ms, ss), (mt, st) = measures(ks, **kw), measures(kt, **kw)
                a, b = {s for s, _ in ss}, {s for s, _ in st}
                out.append({"date": d, "reading": rd, "task_reading": tr, "variant": vname, "spont": ms["n_seq"], "task": mt["n_seq"],
                            "shared": len(a & b), "sets_differ": a != b,
                            "within_tol": abs(ms["n_seq"] - 13) <= 1.3 and abs(mt["n_seq"] - 17) <= 1.7})
F4A = pd.DataFrame(out)
print(F4A.to_string())
print("main variant, readings within tolerance:", int(F4A[F4A.variant == "main"].within_tol.sum()), "of", int((F4A.variant == "main").sum()))

In [ ]:
# Variant: every ordering of distinct cells (reading a, task reading i).
t0 = time.time()
ALLORD = {}
for d, segs in [("2022-09-15", [("spontaneous", "S1"), ("task", "RT")]),
                ("2022-10-03", [("spontaneous", "S1"), ("spontaneous", "S2"), ("task", "RT")]),
                ("2022-12-06", [("spontaneous", "S1"), ("spontaneous", "S2"), ("task", "RT")])]:
    for c, name in segs:
        x = [f for f in files if (f["subject"], f["date"], f["condition"]) == ("Q", d, c)][0]
        ts = dict(sq.segments(x, "i"))[name]
        sig, info = sq.run_segment(x, ts, "a", np.random.default_rng(7), orders="all")
        ALLORD[(d, c, name)] = {"significant": info["significant"], "main": len(sq.finalize(sig)),
                                "documented dedupe": len(sq.finalize(sig, dedupe="documented"))}
print(pd.DataFrame(ALLORD).T.to_string())
print("in %.0f s" % (time.time() - t0))

**Result: not reproduced.** The only Q SMA session with 40 accepted cells is 2022-09-15. Under every reading of method S the counts are far from 13 and 17: with the code's double-counting step as it runs, 157 to 196 sequences in the spontaneous slice and 176 to 311 in the task slice; with the code's more-than-4 rule, 113 to 155 and 156 to 260; with the double-counting step as documented, 5 and 2 to 5. No reading or variant is within tolerance (0 of 6 main readings). Every ordering of the cells (reading a) gives 979 and 1856 significant patterns, and 2 and 0 after the documented step. The sets of sequences differ between the two conditions under every reading (for example 10 shared of 157 and 211 under reading a, task reading i), as the statement says.

## fig4b-example-three-segments-q

> In another Q SMA session with 18 cells and 20 min of spontaneous recording, 8, 13 and 9 sequences
> were found in the two spontaneous halves and the task, and the sequences differed between all three
> segments.

In [ ]:
durs = {x["date"]: (x["events"].shape[0] - 1) / x["rate_ea"] / 60 for x in files if x["subject"] == "Q" and x["condition"] == "spontaneous"}
cand_b = sorted(x["date"] for x in files if x["subject"] == "Q" and x["condition"] == "spontaneous" and x["events"].shape[1] == 18 and durs[x["date"]] >= 19.9)
print("Q SMA sessions with 18 cells and 20 min or more of spontaneous recording:", {d: round(durs[d], 1) for d in cand_b})
out = []
for d in cand_b:
    for rd in sq.READINGS:
        for tr in ("i", "ii"):
            for vname, kw in VARIANTS.items():
                r = [measures(("Q", "SMA", d, c, t, s, rd), **kw) for c, t, s in
                     [("spontaneous", "i", "S1"), ("spontaneous", "i", "S2"), ("task", tr, "RT")]]
                sets = [{s for s, _ in q[1]} for q in r]
                n = [q[0]["n_seq"] for q in r]
                out.append({"date": d, "reading": rd, "task_reading": tr, "variant": vname, "S1": n[0], "S2": n[1], "RT": n[2],
                            "shared_S1_S2": len(sets[0] & sets[1]), "shared_S1_RT": len(sets[0] & sets[2]), "shared_S2_RT": len(sets[1] & sets[2]),
                            "shared_all": len(sets[0] & sets[1] & sets[2]),
                            "all_differ": sets[0] != sets[1] and sets[0] != sets[2] and sets[1] != sets[2],
                            "within_tol": abs(n[0] - 8) <= 0.8 and abs(n[1] - 13) <= 1.3 and abs(n[2] - 9) <= 0.9})
F4B = pd.DataFrame(out)
print(F4B.to_string())
print("main variant, readings within tolerance:", int(F4B[F4B.variant == "main"].within_tol.sum()), "of", int((F4B.variant == "main").sum()))

**Result: not reproduced.** Two Q SMA sessions have 18 cells and at least 20 min of spontaneous recording (2022-10-03, 21.8 min, and 2022-12-06, 20.4 min). Neither matches 8, 13 and 9 under any reading (0 of 12 main readings within tolerance): 2022-10-03 gives 3, 4 and 12 to 13 sequences, 2022-12-06 gives 331 to 386, 613 to 746 and 111 to 174 (main variant). The documented double-counting step gives 1, 1 and 2 to 4 for 2022-10-03, and 3 to 10, 9 to 10 and 10 to 13 for 2022-12-06. The three slices' sets of sequences differ in every reading, and at most 4 sequences are shared by all three.

## fig4d-sequence-measures-vary

> The number of sequences per cell, their repetition and the proportion of cells in sequences varied
> across conditions and imaging sites.

Per slice: sequences over accepted cells, the median number of repetitions of its sequences, and
the percentage of cells in at least one sequence (Figure 4D legend). Per site, the median over
sessions of each measure, per segment (S1, S2, RT). The paper's Figure 4D table (Zenodo
doi:10.5281/zenodo.14976005) is shown next to this attempt's values, matched by session and segment;
it decides nothing.

In [ ]:
main = ALL[ALL.variant == "main"].copy()
main["site"] = main.subject + " " + main.region
def site_medians(df):
    return df.groupby(["site", "segment"]).agg(sessions=("n_seq", "size"), seq_per_cell=("seq_per_cell", "median"),
                                              median_reps=("median_reps", "median"), pct_cells=("pct_cells", "median"))
dec = []
for rd in sq.READINGS:
    for tr in ("i", "ii"):
        df = main[(main.reading == rd) & ((main.condition == "spontaneous") | (main.task_reading == tr))]
        sm = site_medians(df)
        if rd == "a" and tr == "i":
            print("reading a, task reading i: per-site medians")
            print(sm.round(3).to_string())
        for meas in ("seq_per_cell", "median_reps", "pct_cells"):
            by_seg = sm[meas].unstack("segment")
            dec.append({"reading": rd, "task_reading": tr, "measure": meas,
                        "differs_across_segments": bool((by_seg.nunique(axis=1, dropna=True) > 1).any()),
                        "differs_across_sites": bool((by_seg.nunique(axis=0, dropna=True) > 1).any()),
                        "range": f"{sm[meas].min():.3f} to {sm[meas].max():.3f}"})
DEC = pd.DataFrame(dec)
print(DEC.to_string())
print("readings where every measure differs across segments and across sites:",
      int(DEC.groupby(["reading", "task_reading"]).apply(lambda g: (g.differs_across_segments & g.differs_across_sites).all()).sum()), "of 6")

In [ ]:
Z4, z4_md5 = h.read_zenodo_csv("14976005", "Sequence parameters box plots (fig 4D)_Tidy version.csv")
Z4.columns = ["monkey", "file", "total_cells", "structure", "segment", "n_seq", "seq_per_cell", "median_reps", "pct_cells"]
Z4["subject"] = Z4.monkey.str[0]
Z4["date"] = Z4.file.str.extract(r"(\d{4}_\d{2}_\d{2})")[0].str.replace("_", "-")
Z4["segment"] = Z4.segment.replace({"wRT": "RT"})
for c in ("n_seq", "seq_per_cell", "median_reps", "pct_cells"):
    Z4[c] = pd.to_numeric(Z4[c], errors="coerce")
print("Zenodo Fig. 4D table", z4_md5, len(Z4), "rows")
mine = main[(main.reading == "a") & ((main.condition == "spontaneous") | (main.task_reading == "i"))]
cmp4 = Z4.merge(mine[["subject", "date", "segment", "n_seq", "median_reps", "pct_cells"]], on=["subject", "date", "segment"],
                how="left", suffixes=("_paper", "_here"))
cells_here = {(x["subject"], x["date"]): x["events"].shape[1] for x in files}
cmp4["cells_here"] = [cells_here.get((s, d)) for s, d in zip(cmp4.subject, cmp4.date)]
print(cmp4[["subject", "date", "segment", "total_cells", "cells_here", "n_seq_paper", "n_seq_here", "median_reps_paper", "median_reps_here",
            "pct_cells_paper", "pct_cells_here"]].to_string())
ok = cmp4.dropna(subset=["n_seq_paper", "n_seq_here"])
print("slices in both: %d; Spearman correlation of the sequence counts %.3f; median ratio here / paper %.2f"
      % (len(ok), stats.spearmanr(ok.n_seq_paper, ok.n_seq_here)[0], (ok.n_seq_here / ok.n_seq_paper.replace(0, np.nan)).median()))
docu = ALL[(ALL.variant == "documented dedupe") & (ALL.reading == "a") & ((ALL.condition == "spontaneous") | (ALL.task_reading == "i"))]
ok2 = Z4.merge(docu[["subject", "date", "segment", "n_seq"]], on=["subject", "date", "segment"], suffixes=("_paper", "_doc")).dropna()
print("documented dedupe variant: Spearman %.3f; median ratio %.2f" % (stats.spearmanr(ok2.n_seq_paper, ok2.n_seq_doc)[0],
      (ok2.n_seq_doc / ok2.n_seq_paper.replace(0, np.nan)).median()))

**Result: reproduced.** Under all six readings (three of the 0.05 s rules, two of the task slice) each measure's per-site medians differ across segments and across sites: sequences per cell range from 0.160 to 8.477, median repetitions from 4.0 to 18.25 and the percentage of cells in sequences from 24.0 to 100.0. The criterion is met by any measure that is not constant, so this says little. The per-slice counts follow the paper's Figure 4D table in rank (Spearman 0.871 over 56 slices) but are a median 3.12 times larger; with the documented double-counting step they are 0.40 times the table's (Spearman 0.852).

## fig4c-sequence-cells-not-clustered

> Cells involved in sequences were spread across the field of view, with no preferred spatial relation
> between them.

Reading (i): centroids of cells in sequences against those of all cells, spontaneous slices, Q SMA
and U M1, all sessions pooled (Figure 4C legend), each coordinate. Reading (ii): distances between
cells that share a sequence against distances between all cell pairs, per session (pairs pooled over
its slices, each pair once), pooled over sessions. Tests: two-sample Kolmogorov-Smirnov and
Mann-Whitney, two-sided, p < 0.05. Main detection variant, task reading (i), for each 0.05 s reading.

In [ ]:
cent = {(x["subject"], x["region"], x["date"]): x["centroid"] for x in files}
F4C = []
for rd in sq.READINGS:
    # (i) centroids, spontaneous, Q SMA and U M1
    for site in [("Q", "SMA"), ("U", "M1")]:
        allc, inseq = [], []
        for x in files:
            if (x["subject"], x["region"]) != site or x["condition"] != "spontaneous":
                continue
            cs = set()
            for name, ts in sq.segments(x, "i"):
                _, seqs = measures((x["subject"], x["region"], x["date"], "spontaneous", "i", name, rd))
                cs |= {c for s, _ in seqs for c in s}
            c0 = x["centroid"]
            allc.append(c0)
            inseq.append(c0[sorted(cs)] if cs else np.zeros((0, 2)))
        allc, inseq = np.vstack(allc), np.vstack(inseq)
        for ax, nm in ((0, "x"), (1, "y")):
            if len(inseq) >= 2:
                F4C.append({"reading": rd, "comparison": f"(i) centroid {nm}, {site[0]} {site[1]}", "n_seq_cells": len(inseq), "n_all": len(allc),
                            "KS_p": stats.ks_2samp(inseq[:, ax], allc[:, ax]).pvalue,
                            "MW_p": stats.mannwhitneyu(inseq[:, ax], allc[:, ax]).pvalue,
                            "median_seq": np.median(inseq[:, ax]), "median_all": np.median(allc[:, ax])})
    # (ii) distances, every session
    dseq, dall = [], []
    for key3, c0 in cent.items():
        pairs = set()
        for k in RES:
            if k[:3] == key3 and k[6] == rd and (k[3] == "spontaneous" or k[4] == "i"):
                _, seqs = measures(k)
                for s, _ in seqs:
                    u = sorted(set(s))
                    pairs |= {(a, b) for i, a in enumerate(u) for b in u[i + 1:]}
        n = len(c0)
        dist = lambda a, b: np.hypot(*(c0[a] - c0[b])) * h.UM_PER_PIXEL
        dall += [dist(a, b) for a in range(n) for b in range(a + 1, n)]
        dseq += [dist(a, b) for a, b in pairs]
    F4C.append({"reading": rd, "comparison": "(ii) pair distance (um), all sessions", "n_seq_cells": len(dseq), "n_all": len(dall),
                "KS_p": stats.ks_2samp(dseq, dall).pvalue, "MW_p": stats.mannwhitneyu(dseq, dall).pvalue,
                "median_seq": np.median(dseq), "median_all": np.median(dall)})
F4C = pd.DataFrame(F4C)
print(F4C.round(4).to_string())
print("tests with p < 0.05: %d of %d" % (((F4C.KS_p < 0.05).sum() + (F4C.MW_p < 0.05).sum()), 2 * len(F4C)))

**Result: blocked (methods underspecified).** The readings disagree. Reading (i), the centroids of cells in sequences against those of all cells (spontaneous, Q SMA and U M1), finds no difference under any 0.05 s reading (smallest p 0.6776 over 24 tests). Reading (ii), the distances between cells that share a sequence against all pair distances, finds them shorter under every 0.05 s reading (medians 284 to 296 against 320 um, KS p 0.0001 to 0.0020, Mann-Whitney p 0.0002 or below). The criterion blocks the claim when the readings disagree. Reading (ii) is the comparison the Results describe; its test treats pairs from overlapping sequences as independent, and the detection behind it does not match the paper's counts (fig4a, fig4b).

## Summary

| Claim | Outcome |
|---|---|
| fig4a-example-sequences-q | not reproduced |
| fig4b-example-three-segments-q | not reproduced |
| fig4d-sequence-measures-vary | reproduced (a weak criterion) |
| fig4c-sequence-cells-not-clustered | blocked: methods underspecified |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

Differences from the paper's method. The detection is a port of `identify_sequences.m` to Python, not the MATLAB code itself. The values the paper passed to it (time window, separation, minimum repetitions, sequence lengths) are not recorded; the readings of the claims file are run. The events are the dandiset's, which differ from the paper's per-cell table (first notebook). The counts follow the paper's table in rank but not in size: about three times larger with the double-counting step as the code runs it, and less than half with the step as its comments describe it.

What was found about the dataset and code. The double-counting step of `identify_sequences.m` writes 2 into a logical array, so its test never succeeds and it removes nothing. The paper's Figure 4D table counts 17 cells in Q 2022-10-03 and 11 in Q 2022-12-01, where the dandiset has 18 and 13 accepted cells.